## This notebook serves as a demonstration of the LASP SDTP application

#### Some commands to make things easier to test within this notebook

In [1]:
import json
from lasp_sdtp.database.database_controller import db
from tests import conftest

# Clear out contents of database
db.session.query(db.FileQueue).delete()
db.session.query(db.Transactions).delete()
db.session.query(db.Accounts).delete()
db.session.query(db.Metadata).delete()
db.session.query(db.AvailableFiles).delete()
db.session.commit()

# Insert an account to test with
db.insert_data('accounts', [{
    'username': 'ges_disc',
    'role': 'subscriber',
    'allowed_data_products': 'tsis2',
    'registration_open': True}])

# Add file metadata to database
conftest._add_available_files_entries()

## Configuration

#### Subscribers must configure several parameters prior to using the `lasp_sdtp` application.  To configure a subscriber, set the following parameters in the `subscriber_config` dictionary

In [2]:
from lasp_sdtp.config import subscriber_config

In [3]:
subscriber_config['account_expiration_period'] = 1800  # The number of days the subscriber account will be valid
subscriber_config['checksum_type'] = 'sha256'  # The type of checksum that the subscriber will use to validate responses
subscriber_config['distinguished_name'] = ''  # The distingushed name to use in authentication
subscriber_config['expiration_period'] = 180  # The number of days files will remain in the file queue
subscriber_config['max_num_files'] = 10000  # The maximum number of files to return in file list requests
subscriber_config['num_download_threads'] = 5  # The maximum number of simultaneous downloads
subscriber_config['username'] = 'ges_disc'  # The subscriber username (used in authentication)

# Define the file stream(s) used for file transfer, along with any unique extras and tags
subscriber_config['streams'] = {
    "prod": {
        "extras": {"type": {"type": "str", "default": ""}},
        "tags": {"date": {"type": "str", "default": ""}}
    },
    "dev": {
        "extras": {"was_extrapolated": {"type": "bool", "default": ""}},
        "tags": {"irradiance": {"type": "float", "default": ""}}
    }
}

In [4]:
subscriber_config

{'account_expiration_period': 1800,
 'checksum_type': 'sha256',
 'distinguished_name': '',
 'expiration_period': 180,
 'max_num_files': 10000,
 'num_download_threads': 5,
 'username': 'ges_disc',
 'streams': {'prod': {'extras': {'type': {'type': 'str', 'default': ''}},
   'tags': {'date': {'type': 'str', 'default': ''}}},
  'dev': {'extras': {'was_extrapolated': {'type': 'bool', 'default': ''}},
   'tags': {'irradiance': {'type': 'float', 'default': ''}}}}}

## Register Account

Register a new account.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned.
    
Command: `curl PUT "http://127.0.0.1:8000/register -H "Cert-UID: <certificate>"`

In [5]:
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## List Available Files

Return a list of available files given user-supplied parameters.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with a list of the available files and their metadata.  If no parameters are supplied, then all available files are returned.

Command: `curl GET "http://127.0.0.1:8000/files?param1=value1&param2=value2... -H "Cert-UID: <certificate>"`

#### Supported parameters:
- `stream` (e.g. `prod`)
- `ShortName` (e.g. `TSIS_SC_L2`)
- `version` (e.g. `v01`)
- `date` (e.g.`2022-01-01`)
- `start_date` (e.g. `2022-01-01`)
- `end_date` (e.g. `2022-01-02`)


Note that `date` can only be supplied if `start_date` and/or `end_date` are *not* supplied.  Also, if `start_date` is supplied, then `end_date` *must* be supplied, or vice-versa 

#### Examples

In [6]:
# List all available files
! curl -i -X GET "http://127.0.0.1:8000/files" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:ik6g70yexj2dhro987pruigpfc767u3sjrytrgvuz3451kw1l6pnltkli78ykud9", 
      "data_product_id": "tsis2", 
      "date": "2022-02-13", 
      "expires": "2023-03-01", 
      "fileid": 1089, 
      "name": "tsis2_ssi_L3_c24h_v01_20220103_20220104.txt", 
      "shortname": "TSIS2_SSI_L3_24HR_TXT", 
      "size": 61.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:9hdiakkkdqp9vjsc630038n1b608uifsa7el63zh4t095abacideaqreo38o3dy3", 
      "data_product_id": "tsis2", 
      "date": "2022-02-14", 
      "expires": "2023-03-01", 
      "fileid": 1090, 
      "name": "tsis2_ssi_L3_c24h_v01_20220102_20220103.txt", 
      "shortname": "TSIS2_SSI_L3_24HR_TXT", 
      "size": 61.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:u7fsm4pp9vo4q8j10tu01v3hx7s2xsy2aetoozxfoq84mvealbuko56dksjpg8kg", 
      "data_product_id": "tsis2", 
      "date": "2022-02-15", 
      "e

In [7]:
# List all available files with ShortName of 'TSIS_SC_L2'
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:b55u71p9na52l3dhm0a0sytj1havhm7hilrjhe2s19t5npvo7oshre1d5ss2xvx4", 
      "data_product_id": "tsis2", 
      "date": "2022-02-11", 
      "expires": "2023-03-01", 
      "fileid": 1087, 
      "name": "tsis2_sc_L2_v01_20220101_20220102.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:pey3zrjh8kp1nijzd14cnf1hbb4sq360usz1cbn8zt5rrzurzw80saev44i5u8co", 
      "data_product_id": "tsis2", 
      "date": "2021-12-31", 
      "expires": "2023-03-01", 
      "fileid": 1045, 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:z2lnkowa4okpjwzm3bzy82drdcpabk4vwcbam6pzpg683iarsb70u78ymifbgy9a", 
      "data_product_id": "tsis2", 
      "date": "2022-01-01", 
      "expires": "2023-03-01", 
      "fil

In [8]:
# List all available files with specific ShortName and for specific date
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2&date=2022-01-02" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:s9pemz44k9cylr3lbcjht8niisrjmqm564m732tsdpzftefg9o3pul710tx1ios1", 
      "data_product_id": "tsis2", 
      "date": "2022-01-02", 
      "expires": "2023-03-01", 
      "fileid": 1047, 
      "name": "tsis2_sc_L2_v01_20220104_20220105.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }
  ]
}


In [9]:
# List all available files between range between dates
! curl -i -X GET "http://127.0.0.1:8000/files?start_date=2021-12-31&end_date=2022-01-05" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "foo", 
      "data_product_id": "tsis2", 
      "date": "2022-01-01", 
      "expires": "2022-09-03", 
      "fileid": 12345, 
      "name": "test_cleanup_db.txt", 
      "shortname": "TEST_FILE", 
      "size": 1.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "bar", 
      "data_product_id": "tsis2", 
      "date": "2022-01-01", 
      "expires": "2022-09-03", 
      "fileid": 12346, 
      "name": "test_cleanup_db2.txt", 
      "shortname": "TEST_FILE", 
      "size": 1.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "bop", 
      "data_product_id": "tsis2", 
      "date": "2022-01-01", 
      "expires": "2022-09-03", 
      "fileid": 67890, 
      "name": "test_reporting.txt", 
      "shortname": "TEST_FILE", 
      "size": 1.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "bat", 
      "data_product_id": "tsis2", 
      "date": 

## Get Specific File

Return the contents of a file for a given `fileid`.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with the content of the file.  

Command: `curl GET "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [10]:
! curl -i -X GET "http://127.0.0.1:8000/files/1047" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220104_20220105.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220104_20220105.zip"
}


## Delete Specific File

Remove a specific file from the queue.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned with the content of the file.  

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [11]:
! curl -i -X DELETE "http://127.0.0.1:8000/files/1047" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Delete Range of Files

Remove a range of files from the queue.  The user must supply the `Cert-UID` in the header of the request.  The range of files should be positive integers that increase in value.  If successful, a `204` response is returned.

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid_start>-<fileid_end> -H "Cert-UID: <certificate>"`

In [12]:
# First get a few files so that they are in the queue
! curl -i -X GET "http://127.0.0.1:8000/files/1047" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/1048" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/1049" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 142
SDTP-TransactionID: 493
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Fri, 02 Sep 2022 18:59:13 GMT

{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220104_20220105.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220104_20220105.zip"
}
HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 154
SDTP-TransactionID: 494
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Fri, 02 Sep 2022 18:59:13 GMT

{
  "contents": [
    "File contents for tsis2_tsi_L3_c24h_v01_20220104_20220105.txt"
  ], 
  "filename": "tsis2_tsi_L3_c24h_v01_20220104_20220105.txt"
}
HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 154
SDTP-TransactionID: 495
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Fri, 02 Sep 2022 18:59:14 GMT

{
  "contents": [
    "File contents for tsis2_ssi_L3_c24h_v01_20220101_20220102.txt"
  ], 
  "filename": "tsis2_ssi_L3_c24h_v01_20220101_20220102.txt"
}


In [13]:
# Delete the range of files
! curl -i -X DELETE "http://127.0.0.1:8000/files/1047-1049" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Examples of bad/invalid requests

In [14]:
# Send request to endpoint that doesn't exist
# Expect a 404 response
! curl -i -X GET "http://127.0.0.1:8000/bogus_endpoint" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [15]:
# Try to register a bogus account
# Expect a 401 response
subscriber_config['username'] = 'unkown'
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

401


In [16]:
# Request a file list with bogus parameters
# Expect a 400 response
! curl -i -X GET "http://127.0.0.1:8000/files?bogus_param=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2&bogus_param=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files?date=20210101" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400


In [17]:
# Request a file list with valid parameters but no files returned
# Expect a 200 response with empty file list
! curl -i -X GET "http://127.0.0.1:8000/files?date=1984-04-04" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": []
}


In [18]:
# Request a file that doesn't exist
# Expect a 404 response
! curl -i -X GET "http://127.0.0.1:8000/files/1234567" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [19]:
# Request a file with invalid fileid
# Expect a 400 response
! curl -i -X GET "http://127.0.0.1:8000/files/-1" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/1.5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/9999999999999999" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400
400


In [20]:
# Try to delete a file that doesn't exist
# Expect a 404 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/1234567" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [21]:
# Try to delete a file with invalid fileid
# Expect a 400 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/-1" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/1.5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400


In [22]:
# Try to delete a range of files that don't exist
# Expect a 404 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/1234567-1234569" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [23]:
# Try to delete a range of files with invalid fileids
# Expect a 400 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/foo-bar" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/10-5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/123-foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/123 - 456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/10-10" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400
000
400


In [ ]:
# Send A LOT of requests at once